In [0]:
storage_account_name = "cloudproject60306117"
storage_account_key = "AdxIWpu+RCT/r8Gr8rvVZxi25XMS/T0QnFugMQMzWXp6P7RHljEIpEZooyARZ3VJAEalcNRoGaDu+AStblz2Bw=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
# Path to your blob container and file
file_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/yellow_tripdata_2016-03_fixed.csv"

# Read CSV into Spark DataFrame
raw_df = spark.read.csv(file_path, header=True, inferSchema=True)

raw_df.printSchema()
raw_df.show(5)


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+------------------+---------------

In [0]:
from pyspark.sql.functions import col, unix_timestamp

# Load raw CSV into Spark DataFrame
raw_df = spark.read.csv(
    "abfss://raw@cloudproject60306117.dfs.core.windows.net/yellow_tripdata_2016-03.csv",
    header=True,
    inferSchema=True
)

# Drop missing critical fields
df = raw_df.filter(
    (col("tpep_pickup_datetime").isNotNull()) &
    (col("tpep_dropoff_datetime").isNotNull())
)

# Convert datatypes (Spark infers, but we can cast explicitly)
df = df.withColumn("passenger_count", col("passenger_count").cast("int"))
df = df.withColumn("trip_distance", col("trip_distance").cast("double"))
df = df.withColumn("fare_amount", col("fare_amount").cast("double"))

# Validation rules
df = df.filter(col("trip_distance") >= 0)
df = df.filter(col("fare_amount") >= 0)
df = df.filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))

# Transformation: trip duration in minutes
df = df.withColumn(
    "trip_duration_minutes",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60
)

print("Rows after cleaning:", df.count())

# Save to Silver layer
df.write.mode("overwrite").parquet("abfss://processed@cloudproject60306117.dfs.core.windows.net/clean_trips/")


Rows after cleaning: 12193830
